# V3-7 — A6: BiLSTM + additive attention + FFN

This CPU-friendly experiment reuses the frozen A2-MP-HN1 RGB feature cache. It does not decode MP4 files again. The final reported metric remains full-MP4 / video-level, using frozen `top3_mean` aggregation.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_a6_bilstm_additive_attention import (
    Config, build_context, cache_preflight, context_report,
    evaluate_best, load_rgb_features, train_model,
)

config = Config()
RUN_TRAINING = True
RUN_EVALUATION = True
print({'data_root': str(config.data_root), 'primary_aggregation': config.primary_aggregation, 'mp4_decoding_required': False})

In [ ]:
# 1) Verify the frozen V3 split and exact feature-cache scope.
context = build_context(config)
print(context_report(context))
display(context['train'].groupby(['training_role', 'video_label']).agg(
    windows=('sequence_id', 'size'), videos=('video_id', 'nunique'), loss_weight=('loss_weight', 'first')
))
assert context['train'].split.eq('train').all()
assert context['validation'].split.eq('validation').all()

In [ ]:
# 2) Cache and model-shape preflight; no MP4 is opened in this step.
preflight = cache_preflight(context)
print(preflight)
assert preflight['mp4_decoding_required'] is False
assert preflight['features_available'] == preflight['features_expected']

In [ ]:
# 3) Train only the BiLSTM + additive-attention + FFN head.
features = load_rgb_features(context)
if RUN_TRAINING:
    training = train_model(context, features)
    print(training)
else:
    print('Training is disabled. Set RUN_TRAINING=True to start the small CPU-only head training.')

In [ ]:
# 4) Evaluate the best checkpoint at full-MP4/video level and save attention diagnostics.
if RUN_EVALUATION:
    evaluation = evaluate_best(context, features)
    print(json.dumps(evaluation['summary'], ensure_ascii=False, indent=2))
    display(evaluation['aggregation_ablation'])
    display(evaluation['attention_summary'])
else:
    print('Evaluation is disabled. Set RUN_EVALUATION=True after a checkpoint exists.')

In [ ]:
# 5) Compare the frozen primary result with the existing development references.
comparison = pd.DataFrame([
    {'model': 'A2-MP full-MP4 reference', 'f1': 0.7375886525, 'recall': 0.8666666667, 'precision': 0.6419753086, 'pr_auc': 0.7226701049},
    {'model': 'D1 RGB + motion, frozen top3', 'f1': 0.7391304348, 'recall': 0.85, 'precision': 0.6538461538, 'pr_auc': 0.7410797351},
])
if config.summary_path.is_file():
    result = json.loads(config.summary_path.read_text(encoding='utf-8'))['primary_metrics_validation_selected_threshold']
    comparison = pd.concat([comparison, pd.DataFrame([{
        'model': 'A6 BiLSTM + additive attention, frozen top3',
        'f1': result['f1'], 'recall': result['recall'], 'precision': result['precision'], 'pr_auc': result['pr_auc'],
    }])], ignore_index=True)
display(comparison.sort_values(['f1', 'pr_auc'], ascending=False))